# Run MoneyPrinterTurbo on Google Colab

This notebook installs [MoneyPrinterTurbo](https://github.com/harry0703/MoneyPrinterTurbo) in an isolated Python 3.11 environment and launches its WebUI. Colab runtimes are temporary, so generated files are removed when the runtime is reset.

## 1. Install MoneyPrinterTurbo

The setup is safe to run again: it clones the repository on the first run and updates it on later runs. Dependencies are installed in the project's virtual environment to avoid conflicts with Colab's preinstalled packages.

In [1]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/harry0703/MoneyPrinterTurbo.git"

# Update an existing checkout so rerunning this cell does not fail during clone.
if (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True
    )
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "uv", "pyngrok"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
# Use the lockfile in an isolated environment without changing Colab's preinstalled packages.
subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True)
print(f"MoneyPrinterTurbo is ready in {REPO_DIR}")

MoneyPrinterTurbo is ready in /content/MoneyPrinterTurbo


## 2. Configure the Colab tunnel

MoneyPrinterTurbo itself does not require ngrok for normal local use. This notebook uses ngrok only because the Streamlit server runs inside a remote Colab runtime.

Create an account and copy your authentication token from the [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). The next cell reads it without displaying or storing it in the notebook.

In [ ]:
3LbM2SX6WAOvw6ZHrxDwkql9zK9_2DhSybBQkhYFvmNe8Jvga

ngrok config add-authtoken 3LbM2SX6WAOvw6ZHrxDwkql9zK9_2DhSybBQkhYFvmNe8Jvga

In [2]:
from getpass import getpass

from pyngrok import ngrok

# Close tunnels from earlier runs before configuring a new one.
ngrok.kill()

ngrok_token = getpass("Enter your ngrok authentication token: ").strip()
if not ngrok_token:
    raise ValueError("An ngrok authentication token is required")
ngrok.set_auth_token(ngrok_token)
del ngrok_token
print("ngrok authentication configured")

Enter your ngrok authentication token: ··········
ngrok authentication configured


## 3. Launch the WebUI

The cell waits for Streamlit to become healthy before creating the public URL. If startup fails, it includes the recent server log in the error. After opening the URL, use **Settings** to configure the required model and media API keys. To use OpenRouter free LLMs, select **OpenRouter**, enter your OpenRouter API key, and keep the default `minimax/minimax-m3:free` model or choose another model from https://openrouter.ai/models?variant=free.

In [3]:
import time
from pathlib import Path
import subprocess
from urllib.error import URLError
from urllib.request import urlopen

from pyngrok import ngrok

# Sanity checks before starting
REPO_DIR = Path("/content/MoneyPrinterTurbo")
if not REPO_DIR.is_dir():
    raise RuntimeError(f"Repo directory does not exist: {REPO_DIR}")
if not (REPO_DIR / "webui" / "Main.py").is_file():
    raise RuntimeError(f"webui/Main.py not found in {REPO_DIR}")

PORT = 8501
LOG_PATH = Path("/content/moneyprinterturbo-webui.log")

# Make this cell safe to rerun by closing the previous process, log handle, and tunnel.
previous_streamlit_proc = globals().get("streamlit_proc")
if previous_streamlit_proc is not None and previous_streamlit_proc.poll() is None:
    previous_streamlit_proc.terminate()
    previous_streamlit_proc.wait(timeout=10)
previous_streamlit_log = globals().get("streamlit_log")
if previous_streamlit_log is not None and not previous_streamlit_log.closed:
    previous_streamlit_log.close()

try:
    ngrok.kill()
except Exception:
    pass

streamlit_log = LOG_PATH.open("w", encoding="utf-8")
streamlit_proc = subprocess.Popen(
    [
        "uv",
        "run",
        "streamlit",
        "run",
        "webui/Main.py",
        f"--server.port={PORT}",
        "--server.address=0.0.0.0",
        "--browser.gatherUsageStats=False",
        "--client.toolbarMode=minimal",
        "--server.showEmailPrompt=False",
    ],
    cwd=REPO_DIR,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
)

# Poll the health endpoint because startup time varies across Colab runtimes.
deadline = time.time() + 90
server_ready = False
while time.time() < deadline:
    if streamlit_proc.poll() is not None:
        break
    try:
        with urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=2) as response:
            server_ready = response.status == 200
    except (URLError, TimeoutError):
        pass
    if server_ready:
        break
    time.sleep(2)

if not server_ready:
    streamlit_log.flush()
    recent_log = LOG_PATH.read_text(encoding="utf-8", errors="replace")[-4000:]
    raise RuntimeError(f"Streamlit failed to start. Recent log:\n{recent_log}")

# Use an explicit IPv4 URL because pyngrok may otherwise route localhost through ::1.
public_tunnel = ngrok.connect(addr=f"http://127.0.0.1:{PORT}", proto="http", bind_tls=True)
print("MoneyPrinterTurbo is ready:")
print(public_tunnel.public_url)
print(f"Server log: {LOG_PATH}")

PyngrokNgrokHTTPError: ngrok client exception, API returned 400: {"error_code":103,"status_code":400,"msg":"invalid endpoint configuration","details":{"err":"yaml: unmarshal errors:\n  line 1: field schemes not found in type config.Endpoint"}}


In [4]:
from pyngrok import ngrok

# Close any previous tunnel
ngrok.kill()

# Create HTTP tunnel
public_tunnel = ngrok.connect(
    addr=8501,
    proto="http"
)

print("MoneyPrinterTurbo URL:")
print(public_tunnel.public_url)

MoneyPrinterTurbo URL:
https://devotee-blatantly-anthill.ngrok-free.dev


In [1]:
import psutil
import shutil
import torch

# System RAM
ram = psutil.virtual_memory()

print("===== SYSTEM RAM =====")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")
print(f"Used RAM: {ram.used / (1024**3):.2f} GB")
print(f"Available RAM: {ram.available / (1024**3):.2f} GB")

# Storage
disk = shutil.disk_usage("/content")

print("\n===== COLAB STORAGE =====")
print(f"Total: {disk.total / (1024**3):.2f} GB")
print(f"Used: {disk.used / (1024**3):.2f} GB")
print(f"Free: {disk.free / (1024**3):.2f} GB")

# GPU
print("\n===== GPU =====")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Free VRAM: {torch.cuda.mem_get_info()[0] / (1024**3):.2f} GB")
else:
    print("No GPU available")

===== SYSTEM RAM =====
Total RAM: 12.67 GB
Used RAM: 1.00 GB
Available RAM: 11.36 GB

===== COLAB STORAGE =====
Total: 107.72 GB
Used: 0.08 GB
Free: 107.62 GB

===== GPU =====
No GPU available


In [1]:
!nvidia-smi

Thu Nov  5 18:29:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

    x = torch.randn(4096, 4096, device="cuda")
    y = x @ x
    print("GPU computation successful:", y.shape)

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB
GPU computation successful: torch.Size([4096, 4096])


In [3]:
import torch
import sys
import importlib.util

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

packages = [
    "diffusers",
    "transformers",
    "accelerate",
    "safetensors",
    "PIL",
    "cv2",
    "moviepy",
    "gradio",
]

print("\n===== INSTALLED PACKAGES =====")

for package in packages:
    print(package, ":",
          "Installed" if importlib.util.find_spec(package) else "Not installed")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA: 13.0

===== INSTALLED PACKAGES =====
diffusers : Installed
transformers : Installed
accelerate : Installed
safetensors : Installed
PIL : Installed
cv2 : Installed
moviepy : Installed
gradio : Installed
